In [ ]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

!pip install transformers datasets scikit-learn -q
import pandas as pd
import numpy as np
import torch

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    IntervalStrategy
)

from datasets import Dataset

In [ ]:
df = pd.read_csv("claim_matching_dataset.csv")


import numpy as np

def get_majority_label(row):
    labels = []

    for col in ["annotator1", "annotator2", "annotator3"]:
        val = str(row[col]).strip()
        if val != "nan" and "N/A" not in val:
            labels.append(val)

    if len(labels) == 0:
        return np.nan

    return max(set(labels), key=labels.count)

df["majority_label"] = df.apply(get_majority_label, axis=1)
df = df.dropna(subset=["majority_label"])


df = df[["text1", "text2", "majority_label"]]
df = df.dropna()

df.head()

,text1,text2,majority_label
0,ആളെ കിട്ടിയിട്ടുണ്ട്... ഇവനാണ് പമ്പ നിലക്കലിൽ ...,രാഹുൽ ഗാന്ധിയുടെ സ്ഥാനാർത്ഥിത്വം പ്രഖ്യാപിച്ചപ...,Very Dissimilar
1,ദുൽഖർ സൽമാൻ കൊണ്ടോട്ടിയിലെ സേവാഭാരതി ക്യാമ്പി...,ആര്‍.എസ്.എസ് കാര്യാലയത്തില്‍ പൊലീസ് റെയ്ഡ്. ന...,Very Dissimilar
2,സേവാഭാരതി സേവാകേന്ദ്രം പുവ്വാട്ടുപറമ്പിൽ ശ്രീ ...,ഇങ്ങനെ തന്നെ നിങ്ങൾ പ്രചരിപ്പിക്കണം അപ്പഴേ ആളു...,Very Dissimilar
3,. *_SKY ViSiON_* / / ...,കേരളത്തിലെ ആം ആദ്മിയുടെ വിജയകരമായ ഒരു attempt....,Very Dissimilar
4,. *_SKY VISION_* / / ...,വണ്ടിക്ക് മുന്നിൽ നിന്ന് നില്ല്.. നില്ല്..എന്ന...,Very Dissimilar


In [ ]:
print(df.columns)
df.head()

Index(['text1', 'text2', 'majority_label'], dtype='object')


,text1,text2,majority_label
0,ആളെ കിട്ടിയിട്ടുണ്ട്... ഇവനാണ് പമ്പ നിലക്കലിൽ ...,രാഹുൽ ഗാന്ധിയുടെ സ്ഥാനാർത്ഥിത്വം പ്രഖ്യാപിച്ചപ...,Very Dissimilar
1,ദുൽഖർ സൽമാൻ കൊണ്ടോട്ടിയിലെ സേവാഭാരതി ക്യാമ്പി...,ആര്‍.എസ്.എസ് കാര്യാലയത്തില്‍ പൊലീസ് റെയ്ഡ്. ന...,Very Dissimilar
2,സേവാഭാരതി സേവാകേന്ദ്രം പുവ്വാട്ടുപറമ്പിൽ ശ്രീ ...,ഇങ്ങനെ തന്നെ നിങ്ങൾ പ്രചരിപ്പിക്കണം അപ്പഴേ ആളു...,Very Dissimilar
3,. *_SKY ViSiON_* / / ...,കേരളത്തിലെ ആം ആദ്മിയുടെ വിജയകരമായ ഒരു attempt....,Very Dissimilar
4,. *_SKY VISION_* / / ...,വണ്ടിക്ക് മുന്നിൽ നിന്ന് നില്ല്.. നില്ല്..എന്ന...,Very Dissimilar


In [ ]:
label_map = {
    "Very Similar": 0,
    "Somewhat Similar": 1,
    "Somewhat Dissimilar": 2,
    "Very Dissimilar": 3
}

df["label"] = df["majority_label"].map(label_map)

# Remove rows where label might be NaN after mapping and convert to integer
df = df.dropna(subset=["label"])
# Use .loc to explicitly assign the converted integer values to avoid SettingWithCopyWarning
df.loc[:, "label"] = df["label"].astype(int)

In [ ]:
print(df["label"].value_counts())

label
3.0    1750
0.0     254
1.0     215
2.0     113
Name: count, dtype: int64


In [ ]:

model_name = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(example):
    return tokenizer(
        example["text1"],
        example["text2"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

dataset = Dataset.from_pandas(df_balanced)
dataset = dataset.map(tokenize_function, batched=True)
dataset = dataset.rename_column("label", "labels")
dataset = dataset.map(
    lambda ex: {"labels": [int(x) for x in ex["labels"]]},
    batched=True
)
print("Dataset ready:", dataset)

config.json:   0%|          | 0.00/723 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/402 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Map:   0%|          | 0/1800 [00:00<?, ? examples/s]

Map:   0%|          | 0/1800 [00:00<?, ? examples/s]

Dataset ready: Dataset({
    features: ['text1', 'text2', 'majority_label', 'labels', 'input_ids', 'attention_mask'],
    num_rows: 1800
})


In [ ]:
def tokenize_function(example):
    return tokenizer(
        example["text1"],
        example["text2"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

In [ ]:
dataset = Dataset.from_pandas(df)

dataset = dataset.map(tokenize_function, batched=True)

dataset = dataset.rename_column("label", "labels")

# Explicitly cast 'labels' to torch.long type
dataset = dataset.map(lambda examples: {"labels": [torch.tensor(label, dtype=torch.long) for label in examples["labels"]]}, batched=True)


dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)

dataset

Map:   0%|          | 0/2332 [00:00<?, ? examples/s]

Map:   0%|          | 0/2332 [00:00<?, ? examples/s]

Dataset({
    features: ['text1', 'text2', 'majority_label', 'labels', '__index_level_0__', 'input_ids', 'attention_mask'],
    num_rows: 2332
})

In [ ]:
kf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

X = df.index
y = df["label"]

f1_scores = []

In [ ]:
from collections import Counter
from sklearn.utils.class_weight import compute_class_weight
import numpy as np
import torch

print("Before balancing:", Counter(df["label"]))

df_majority = df[df["label"] == 3]
df_c0 = df[df["label"] == 0]
df_c1 = df[df["label"] == 1]
df_c2 = df[df["label"] == 2]

df_c0_up = df_c0.sample(400, replace=True, random_state=42)
df_c1_up = df_c1.sample(400, replace=True, random_state=42)
df_c2_up = df_c2.sample(400, replace=True, random_state=42)
df_majority_down = df_majority.sample(600, random_state=42)

df_balanced = pd.concat([df_majority_down, df_c0_up, df_c1_up, df_c2_up])
df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

print("After balancing:", Counter(df_balanced["label"]))

clean_labels = df_balanced["label"].astype(int).values
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(clean_labels),
    y=clean_labels
)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float)
print("Class weights:", class_weights_tensor)

Before balancing: Counter({3.0: 1750, 0.0: 254, 1.0: 215, 2.0: 113})
After balancing: Counter({3.0: 600, 2.0: 400, 0.0: 400, 1.0: 400})
Class weights: tensor([1.1250, 1.1250, 1.1250, 0.7500])


In [ ]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning)
from torch import nn
from transformers import Trainer, TrainingArguments, AutoModelForSequenceClassification
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
import numpy as np
import torch


class WeightedTrainer(Trainer):
    def __init__(self, class_weights, **kwargs):
        super().__init__(**kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss = nn.CrossEntropyLoss(
            weight=self.class_weights.to(logits.device)
        )(logits, labels)
        return (loss, outputs) if return_outputs else loss


def collate_fn(batch):
    return {
        "input_ids": torch.stack([
            x["input_ids"].clone().detach() if torch.is_tensor(x["input_ids"])
            else torch.tensor(x["input_ids"], dtype=torch.long)
            for x in batch
        ]),
        "attention_mask": torch.stack([
            x["attention_mask"].clone().detach() if torch.is_tensor(x["attention_mask"])
            else torch.tensor(x["attention_mask"], dtype=torch.long)
            for x in batch
        ]),
        "labels": torch.tensor([
            x["labels"].item() if torch.is_tensor(x["labels"]) else int(x["labels"])
            for x in batch
        ], dtype=torch.long),
    }


kf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
X = df_balanced.index
y = df_balanced["label"]
f1_scores = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X, y)):
    print(f"\n===== Fold {fold+1} =====")

    train_dataset = dataset.select(train_idx)
    val_dataset = dataset.select(val_idx)

    train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
    val_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=4,
        problem_type="single_label_classification"
    )

    # ---- UPDATED Training Arguments ----
    training_args = TrainingArguments(
        output_dir=f"./results_fold_{fold}",
        eval_strategy="epoch",
        save_strategy="no",
        learning_rate=3e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        num_train_epochs=4,
        warmup_steps=50,
        weight_decay=0.01,
        report_to="none"
    )

    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        preds = np.argmax(logits, axis=1)
        return {"f1": f1_score(labels, preds, average="macro")}

    trainer = WeightedTrainer(
        class_weights=class_weights_tensor,
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        compute_metrics=compute_metrics,
        data_collator=collate_fn
    )

    trainer.train()
    eval_result = trainer.evaluate()
    print(f"Fold {fold+1} F1:", eval_result["eval_f1"])
    f1_scores.append(eval_result["eval_f1"])

print("\nFinal Mean F1 Score:", np.mean(f1_scores))


===== Fold 1 =====


model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: sentence-transformers/paraphrase-multilingual-mpnet-base-v2
Key                        | Status     | 
---------------------------+------------+-
pooler.dense.weight        | UNEXPECTED | 
pooler.dense.bias          | UNEXPECTED | 
embeddings.position_ids    | UNEXPECTED | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,F1
1,No log,0.862564,0.217248
2,No log,0.779540,0.322816
3,No log,0.717384,0.406593


In [ ]:
torch.autograd.set_detect_anomaly(True)

In [1]:
print("Final F1 Score:", np.mean(f1_scores))

NameError: name 'np' is not defined

In [ ]:
all_runs = []

for run in range(3):
    print(f"\n######## RUN {run+1} ########")
    f1_scores = []

    for train_idx, val_idx in kf.split(X, y):
        train_dataset = dataset.select(train_idx)
        val_dataset = dataset.select(val_idx)

        model = AutoModelForSequenceClassification.from_pretrained(
            model_name,
            num_labels=4
        )

        trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=train_dataset,
            eval_dataset=val_dataset,
            tokenizer=tokenizer,
            compute_metrics=compute_metrics
        )

        trainer.train()
        eval_result = trainer.evaluate()
        f1_scores.append(eval_result["eval_f1"])

    all_runs.append(np.mean(f1_scores))

print("Final Avg F1:", np.mean(all_runs))